# Validación del Operador Categórico y Equivalencia Binaria (Anexo E)

Este cuaderno contiene la validación empírica y los análisis inferenciales (tests de Wilcoxon) de los operadores discretos:

1. **Estudio 1: Softmax Continuo (Legacy) vs. Operador de Salto Modular Unificado**
   - Evaluación en 5 paisajes con 15 réplicas independientes (75 contrastes).
   - Test pareado de rangos con signo de Wilcoxon por paisaje.
2. **Estudio 2: Equivalencia Formal entre Variable Binaria (XOR) y Categórica ($K=2$)**
   - Evaluación sobre 50 semillas estadísticas con $N=32$ variables.
   - Test de Wilcoxon ($p = 0.6547$) demostrando ausencia de diferencias significativas.

Los datos se almacenan en formato dual:
- `data/*.npz`: Archivos binarios de NumPy comprimidos para arrays de pérdidas y trayectorias.
- `data/*.json`: Resúmenes estadísticos legibles y estructurados.


In [1]:
import os
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display

# Rutas del experimento
cwd = Path.cwd().resolve()
project_root = next(p for p in [cwd] + list(cwd.parents) if (p / "src").is_dir())
import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
DATA_DIR = project_root / "experiments" / "extra" / "categorical" / "data"
RESULTS_DIR = DATA_DIR
FIGURES_DIR = project_root / "experiments" / "extra" / "categorical" / "figures"

# User execution control:
# - Set RUN_SIMULATION = True to re-run the categorical benchmarks and statistical tests from scratch.
# - Set RUN_SIMULATION = False to load precomputed results from data/.
RUN_SIMULATION = False

print(f">>> Categorical Experiment | RUN_SIMULATION={RUN_SIMULATION}")

if RUN_SIMULATION:
    print(">>> RUN_SIMULATION=True: Executing categorical benchmarks from scratch...")
    from experiments.extra.categorical.runners.run_experiments import main as run_cat_main
    run_cat_main()


>>> Categorical Experiment | RUN_SIMULATION=False


In [2]:
# Verificación de los archivos binarios NumPy (.npz) generados
npz_cat = np.load(os.path.join(RESULTS_DIR, 'categorical_comparison_data.npz'))
npz_bin = np.load(os.path.join(RESULTS_DIR, 'binary_vs_categorical_data.npz'))
print('Archivos NumPy (.npz) disponibles:')
print('  [categorical_comparison_data.npz]:', list(npz_cat.files)[:5], f'... ({len(npz_cat.files)} arrays)')
print('  [binary_vs_categorical_data.npz]:  ', list(npz_bin.files))


---
## 1. Estudio 1: Softmax Legacy vs. Operador de Salto Modular

Contrastamos el modelo anterior ($K$ logits con muestreo Softmax continuo *memoryless*) frente al nuevo operador unificado ($c_i \in \{0, \dots, K_i-1\}$ con autoadaptación de 1 logit de probabilidad de mutación $p_{\text{mut}} = \sigma(z_i)$).


In [ ]:
# Cargar y presentar métricas y tests de Wilcoxon del Estudio 1
with open(os.path.join(DATA_DIR, 'categorical_comparison_results.json'), 'r', encoding='utf-8') as f:
    s1_data = json.load(f)

header = f"{'Banco de pruebas':<34} | {'Legacy (Softmax)':<17} | {'Nuevo (Salto)':<17} | {'W/T/L':<7} | {'Wilcoxon p':<12} | {'Speedup':<7}"
print(header)
print('-' * len(header))
for b_id, d in s1_data.items():
    leg_s = f"{d['legacy_loss_mean']:.2f} ± {d['legacy_loss_std']:.2f} ({d['legacy_success_pct']:.0f}%)"
    new_s = f"{d['new_loss_mean']:.2f} ± {d['new_loss_std']:.2f} ({d['new_success_pct']:.0f}%)"
    rec = f"{d['wins']}/{d['ties']}/{d['losses']}"
    p_str = f"{d['wilcoxon_p']:.2e}"
    spd = f"{d['speedup']:.1f}x"
    print(f"{d['name']:<34} | {leg_s:<17} | {new_s:<17} | {rec:<7} | {p_str:<12} | {spd:<7}")


### Dinámica de Convergencia en los 5 Paisajes
El modelo anterior queda atrapado por el ruido de muestreo de Softmax, mientras que el nuevo operador converge suavemente y sin oscilaciones residuales.


In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, 'categorical_convergence.png')))


---
## 2. Estudio 2: Equivalencia Formal Binaria vs. Categórica ($K=2$)

Contrastamos si la variable categórica con $K=2$ y la variable binaria (XOR) son equivalentes estocásticamente a lo largo de 50 semillas independientes en $N=32$ variables (5.000 evaluaciones por réplica).


In [ ]:
# Cargar y presentar métricas del Estudio 2
with open(os.path.join(RESULTS_DIR, 'binary_vs_categorical_k2.json'), 'r', encoding='utf-8') as f:
    s2_data = json.load(f)

print(f"Réplicas analizadas:           {s2_data['n_seeds']} semillas")
print(f"Pérdida media Binaria (XOR):    {s2_data['binary_loss_mean']:.4f} ± {s2_data['binary_loss_std']:.4f} (Éxito: {s2_data['binary_success_pct']:.1f}%)")
print(f"Pérdida media Categórica (K=2): {s2_data['categorical_loss_mean']:.4f} ± {s2_data['categorical_loss_std']:.4f} (Éxito: {s2_data['categorical_success_pct']:.1f}%)")
print(f"Semillas con resultado idéntico: {s2_data['identical_seeds_count']}/50 ({s2_data['identical_seeds_count']*2}%)")
print('-' * 70)
print(f"Wilcoxon signed-rank test:      W = {s2_data['wilcoxon_stat']:.1f}, p = {s2_data['wilcoxon_p']:.4f}")
print(f"Mann-Whitney U test:            p = {s2_data['mann_whitney_u_p']:.4f}")
print(f"Test exacto de Fisher:          p = {s2_data['fisher_exact_p']:.4f}")

if s2_data['wilcoxon_p'] > 0.05:
    print('\n>> CONCLUSIÓN ESTADÍSTICA: No existe diferencia significativa (p >> 0.05).')
    print('>> Ambos operadores describen la misma distribución subyacente y son equivalentes.')


In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, 'binary_vs_cat_k2.png')))


---
## 3. Conclusiones para el Anexo E del TFG

1. **Resolución del Ruido de Muestreo:** El operador discreto unificado elimina por completo el suelo de ruido asintótico del modelo continuo Softmax ($p < 10^{-3}$ en el test de Wilcoxon en todos los problemas).
2. **Eficiencia Temporal:** La reducción de complejidad algorítmica de $\mathcal{O}(N \cdot K)$ a $\mathcal{O}(N)$ y la eliminación de cálculos de exponenciales y ruletas a cada paso proporciona una aceleración de **$7\times$ a $12.7\times$** en CPU.
3. **Isomorfismo Formal:** Para $K=2$, la mutación categórica por salto $(c + 1) \pmod 2$ es isomórfica al flip binario $b \oplus 1$. Los tests no paramétricos pareados confirman la equivalencia estadística ($p = 0.6547$).
